# Diagnóstico de memória — pipeline NAS

Corre no **mesmo kernel** que o pipeline, por isso mede o ambiente real,
incluindo o que o próprio Jupyter retém.

Por cada run mostra:

* **RSS do processo** — o número que interessa
* **tamanho dos containers do `api`** do NATS-Bench — o principal suspeito
* **a nossa cache** de memoização, para confirmar que está limitada
* **que tipos Python mais cresceram**

A última célula sonda se é possível despejar a cache interna da API, que é a
correcção que quero aplicar a seguir.

> Se a RSS subir **a par** de um container da API, está identificado.
> Se subir sem nada crescer do lado do Python, é memória nativa ou fragmentação
> e a saída é outra (isolar cada run num subprocesso).

## 1. Setup

In [ ]:
import ctypes, gc, io, contextlib, os, sys
from collections import Counter

ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, 'utils')):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)
print('ROOT       :', ROOT)
print('interpreter:', sys.executable, sys.version_info[:2])


def rssMiB():
    """Working set do processo, sem depender do psutil."""
    if os.name == 'nt':
        import ctypes.wintypes as wt

        class PMC(ctypes.Structure):
            _fields_ = [('cb', wt.DWORD), ('PageFaultCount', wt.DWORD),
                        ('PeakWorkingSetSize', ctypes.c_size_t),
                        ('WorkingSetSize', ctypes.c_size_t),
                        ('QuotaPeakPagedPoolUsage', ctypes.c_size_t),
                        ('QuotaPagedPoolUsage', ctypes.c_size_t),
                        ('QuotaPeakNonPagedPoolUsage', ctypes.c_size_t),
                        ('QuotaNonPagedPoolUsage', ctypes.c_size_t),
                        ('PagefileUsage', ctypes.c_size_t),
                        ('PeakPagefileUsage', ctypes.c_size_t)]
        c = PMC(); c.cb = ctypes.sizeof(PMC)
        ctypes.windll.psapi.GetProcessMemoryInfo(
            ctypes.windll.kernel32.GetCurrentProcess(), ctypes.byref(c), c.cb)
        return c.WorkingSetSize / 2**20
    # Linux (o container): VmRSS é o valor corrente, ru_maxrss só o pico
    try:
        with open('/proc/self/status') as fh:
            for line in fh:
                if line.startswith('VmRSS:'):
                    return int(line.split()[1]) / 1024
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024


def apiContainers(api, top=6):
    """Os maiores atributos dict/list/set pendurados no objecto api."""
    out = []
    for name in dir(api):
        if name.startswith('__'):
            continue
        try:
            v = getattr(api, name)
        except Exception:
            continue
        if isinstance(v, (dict, list, set, tuple)) and len(v):
            out.append((name, len(v)))
    return sorted(out, key=lambda x: -x[1])[:top]


def typeCounts():
    c = Counter()
    for o in gc.get_objects():
        c[type(o).__name__] += 1
    return c


print('RSS agora  :', f'{rssMiB():.0f} MiB')

## 2. Carregar a API e medir a linha de base

In [ ]:
from utils.nas import getApi
import utils.nas as nas
import utils.nas_mo as mo

NAS_PATH = os.path.join(ROOT, 'utils', 'nas_records',
                        'NATS-tss-v1_0', 'NATS-tss-v1_0')
assert os.path.isdir(NAS_PATH), f'benchmark nao encontrado em {NAS_PATH}'

before = rssMiB()
api = getApi(NAS_PATH)
gc.collect()
print(f'RSS antes da API : {before:.0f} MiB')
print(f'RSS depois       : {rssMiB():.0f} MiB')
print('containers da API:', apiContainers(api))
print('clear_params existe?', hasattr(api, 'clear_params'))

## 3. Runs sucessivos

`N_RUNS = 8` num só dataset chega para ver a tendência. O valor absoluto vai ser
muito menor do que o que te matou a máquina — o que interessa é **se o container
da API cresce monotonamente**. Se crescer, extrapola sozinho para os ~470 runs
do sweep completo.

In [ ]:
N_RUNS  = 8
DATASET = 'cifar10'
FMIN, FMAX = 7.78305, 220.11969        # CIFAR-10; irrelevante para a memória

base_types = typeCounts()
prev = rssMiB()
rows = []

print(f'{"run":>4} | {"RSS MiB":>8} | {"delta":>7} | {"cache":>7} | containers da API')
print('-' * 96)

for k in range(1, N_RUNS + 1):
    with contextlib.redirect_stdout(io.StringIO()):
        mo.runMONASScalarized(
            'GA', api, dataset=DATASET, hp='200', alpha=0.5,
            n_generations=50, pop_size=20, seed=k,
            flops_min=FMIN, flops_max=FMAX,
            results_dir=os.path.join(ROOT, '_diag_out'), verbose=False)
    gc.collect()
    cur = rssMiB()
    conts = apiContainers(api, top=3)
    rows.append((k, cur, len(nas._QUERY_CACHE), conts))
    print(f'{k:>4} | {cur:8.0f} | {cur - prev:+7.1f} | '
          f'{len(nas._QUERY_CACHE):>7} | {conts}')
    prev = cur

## 4. Que tipos cresceram

In [ ]:
gc.collect()
now = typeCounts()
grew = sorted(((now[t] - base_types.get(t, 0), t) for t in now), reverse=True)
print('tipo                                   crescimento')
for d, t in grew[:12]:
    if d > 0:
        print(f'  {t:<36} {d:>10,}')

## 5. É possível despejar a cache interna da API?

Esta é a correcção que quero aplicar: depois de copiarmos os dados para a nossa
cache, a cópia da API é peso morto. A célula identifica o atributo certo,
despeja uma entrada à experiência e volta a consultá-la, para confirmar que a
API a recarrega sem rebentar.

In [ ]:
print('classe da API:', ' -> '.join(c.__name__ for c in type(api).__mro__[:3]))
for name in ('arch2infos_dict', 'evaluated_indexes', 'arch2infos_full',
             'arch2infos_less', '_all_infos'):
    v = getattr(api, name, None)
    kind = type(v).__name__ if v is not None else '-'
    size = len(v) if isinstance(v, (dict, list, set)) else ''
    print(f'  {name:20} {kind:<12} {size}')

store = getattr(api, 'arch2infos_dict', None)
if not isinstance(store, dict) or not store:
    print('\n-> arch2infos_dict nao encontrado/vazio.')
    print('   Diz-me qual dos containers da tabela acima cresceu e ataco esse.')
else:
    victim = next(iter(store))
    before_n = len(store)
    ev = getattr(api, 'evaluated_indexes', None)
    store.pop(victim, None)
    if isinstance(ev, set):
        ev.discard(victim)
    elif isinstance(ev, list) and victim in ev:
        ev.remove(victim)
    print(f'\ndespejei o indice {victim}: {before_n} -> {len(store)} entradas')

    try:
        nas.clearQueryCache()
        r = nas.queryValidTest(api, [0, 0, 0, 0, 0, 0], dataset='cifar10', hp='200')
        print(f'consulta apos despejo: OK  (test_acc={r.get("test_acc")})')
        print(f'store voltou a {len(store)} entradas (recarregou sob procura)')
        print('\n-> DESPEJO VIAVEL: posso limitar a memoria da API a um run.')
    except Exception as e:
        print(f'consulta apos despejo REBENTOU: {type(e).__name__}: {e}')
        print('\n-> nao viavel por esta via; uso recriacao periodica da API.')

## 6. Como ler

| Observação | Causa | Correcção |
|---|---|---|
| RSS sobe **e** um container da API cresce junto | `arch2infos_dict` sem despejo | despejar após copiarmos (célula 5 diz se dá) |
| RSS sobe e **nada** em Python cresce | memória nativa / fragmentação | isolar cada run num subprocesso |
| RSS estável aqui mas cresce no pipeline completo | outro dataset, ou o histórico do Jupyter | comparar com `python diagnose_memory.py` no terminal |

Cola-me a saída das secções 3 e 5.